# A project's TRE reads the catalog of a Data Provider

Run from a notebook of the TRE (`cvd-2026`), signed in as a member of the project. It checks, one step at a time, what a member needs to read the `cvd`
catalog of the Data Provider `lon`, and says which step fails, so that the failure says where to look:

1. the token that the notebook holds names the project, the Data Provider as an audience, and the second factor;
2. the notebook pod cannot call `lon`'s Polaris directly, as the TRE's whitelist has it: only Spark can;
3. Spark reads the table that `lon` has released, and gets its columns;
4. Spark is refused a table that `lon` has not released.

Needs two tables on `lon`, which its own smoke test writes (`afon-cli.sh manage-project run lon spark-polaris-smoke`): `cvd.staging.smoke_test`, which `lon` then releases
(`afon-cli.sh manage-project run lon manage-project-grants release cvd smoke_test`), and `cvd.staging.smoke_test_unreleased`, which stays in `staging`.
Change the cell below for another Data Provider, catalog or table.

In [ ]:
# What is read, and where.  The endpoint is that of the Data Provider's Polaris, as its catalog has it in the TRE's config (AZURE_CATALOG_URI_lon_cvd)
PROJECT       = "cvd-2026"
AUDIENCE      = "provider-lon"                                   # the audience of the Data Provider
CATALOG_URI   = "https://azure-polaris.lon.emrys2.click/api/catalog"
CATALOG_NAME  = "cvd"                                            # the name of the catalog in the Data Provider's Polaris
SPARK_CATALOG = "test_lon_cvd"                                   # what it is called here
RELEASED      = "released"                                       # the namespace that the project may read
NOT_RELEASED  = "staging"                                        # the namespace that it may not
TABLE         = "smoke_test"                                     # released by the Data Provider
UNRELEASED_TABLE = "smoke_test_unreleased"                       # made in NOT_RELEASED, and not released: it must exist there
EXPECTED_COLUMNS = ["value"]
SPARK_CONNECT = "sc://spark-connector-0-driver-svc:15002"

results = {}

def record(step, ok, detail=""):
    """ok is True (pass), False (fail), or None: not conclusive, which the result says but does not count as a failure"""
    results[step] = ok
    label = "PASS" if ok is True else "FAIL" if ok is False else "INCONCLUSIVE"
    print(label, "-", step, ("- " + detail) if detail else "")

## 1. The token

`OIDC_TOKEN` is put in the environment of the notebook when it starts, from the member's sign-in.  Only the claims are printed, never the token.

In [ ]:
import os, json, base64, time

token = os.environ.get("OIDC_TOKEN")
if not token:
    record("token is in the environment", False, "OIDC_TOKEN is not set: sign out and in again, or the hub does not pass it on")
else:
    payload = token.split(".")[1]
    claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
    for name in ("iss", "aud", "azp", "acr", "project", "polaris_principal", "email"):
        print(f"{name} = {claims.get(name)}")
    remaining = claims["exp"] - time.time()
    record("token names the project", PROJECT in (claims.get("project") or []), f"project = {claims.get('project')}")
    record("token is for the Data Provider", AUDIENCE in (claims.get("aud") or []), f"aud = {claims.get('aud')}")
    record("token says a second factor was used", claims.get("acr") == "mfa", f"acr = {claims.get('acr')}")
    record("token has not expired", remaining > 60, f"{int(remaining)} seconds left")

## 2. The notebook cannot call the Data Provider directly

A member reaches the Data Provider's catalog through Spark, whose driver and workers are allowed its hosts by the TRE's whitelist; the notebook pod is not: its
network policy admits only Spark.  So a call of the catalog API from here, outside Spark, must not get through.  Not resolving the name, or not connecting, is
the result that is wanted: if the call is answered, the notebook has a way out that the whitelist does not mean it to have.

In [ ]:
import urllib.request, urllib.error, socket
from urllib.parse import urlparse

STEP = "the notebook pod cannot call the Data Provider directly"
host = urlparse(CATALOG_URI).hostname
try:
    address = socket.gethostbyname(host)
except OSError as e:
    record(STEP, True, f"{host} does not resolve from here, as the network policy of the notebook pods has it: {e}")
    address = None

if address:
    request = urllib.request.Request(f"{CATALOG_URI}/v1/config?warehouse={CATALOG_NAME}", headers={"Authorization": "Bearer " + token})
    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            record(STEP, False, f"HTTP {response.status}: the notebook reached the Data Provider's Polaris, which its network policy should not allow")
    except urllib.error.HTTPError as e:
        record(STEP, False, f"HTTP {e.code}: the notebook reached the Data Provider's Polaris, which its network policy should not allow")
    except (urllib.error.URLError, socket.timeout, OSError) as e:
        record(STEP, True, f"{host} resolves, and the connection does not get through, as the network policy of the notebook pods has it: {e}")

## 3. Spark session with the Data Provider's catalog

The shared Spark Connect server registers no catalog of its own: each catalog is given here, with the member's token, as the generated session code does.

In [ ]:
from pyspark.sql import SparkSession

builder = (SparkSession.builder.appName("tre-reads-data-provider").remote(SPARK_CONNECT)
    .config(f"spark.sql.catalog.{SPARK_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.type", "rest")
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.uri", CATALOG_URI)
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.warehouse", CATALOG_NAME)
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.token", token)
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.cache-enabled", "false")
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.header.X-Iceberg-Access-Delegation", "vended-credentials"))
try:
    spark = builder.getOrCreate()
    record("Spark session is made", True)
except Exception as e:
    record("Spark session is made", False, str(e)[:300])

## 4. Reading what `lon` has released

In [ ]:
try:
    table = f"{SPARK_CATALOG}.{RELEASED}.{TABLE}"
    frame = spark.sql(f"SELECT * FROM {table}")
    frame.printSchema()
    rows = frame.collect()
    record(f"reads {table}", True, f"{len(rows)} row(s)")
    record("the table has the columns that were made", frame.columns == EXPECTED_COLUMNS, f"columns = {frame.columns}")
except Exception as e:
    record(f"reads {SPARK_CATALOG}.{RELEASED}.{TABLE}", False, str(e)[:600])

## 5. Being refused what `lon` has not released

The project's role is read-only on `released` and has nothing on `staging`, where the Data Provider makes and checks its tables.  `UNRELEASED_TABLE` is a
table that exists there and has not been released, so reading it must be refused, and the refusal must be one of authorization: a table that is not found
says nothing, as it may not be there, or Polaris may hide it from a caller who has no right to it, and is reported as inconclusive.

In [ ]:
table = f"{SPARK_CATALOG}.{NOT_RELEASED}.{UNRELEASED_TABLE}"
DENIED = ("forbidden", "not authorized", "notauthorized", "403", "permission", "not allowed", "denied")
try:
    spark.sql(f"SELECT * FROM {table}").collect()
    record(f"is refused {table}", False, "it was read: the project can read what the Data Provider has not released")
except Exception as e:
    message = str(e)
    first = message.strip().splitlines()[0][:200]
    if any(word in message.lower() for word in DENIED):
        record(f"is refused {table}", True, "refused: " + first)
    elif "not found" in message.lower() or "table_or_view_not_found" in message.lower():
        record(f"is refused {table}", None, "not found, which is not a refusal: check that the table exists in " + NOT_RELEASED + " on the Data Provider, or that Polaris hides it: " + first)
    else:
        record(f"is refused {table}", None, "an error that is not one of authorization: " + first)

## Result

In [ ]:
print()
for step, ok in results.items():
    print("PASS" if ok is True else "FAIL" if ok is False else "INCONCLUSIVE", "-", step)
print()
if any(ok is False for ok in results.values()):
    print("SOMETHING FAILED: the first FAIL above says where")
elif any(ok is None for ok in results.values()):
    print("NO FAILURE, BUT NOT EVERYTHING WAS CONCLUDED: see INCONCLUSIVE")
else:
    print("ALL PASS")